# Testing provenance

## Table of content

- [Create toy dataframes](#create-toy-dataframes)
- [Tests with data_provenance_enabled](#tests-with-data_provenance_enabled)
    - [Select](#select)
    - [Join](#join)
    - [Where](#where)
    - [Aggregation](#aggregation)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#%pip install pyspark


In [ ]:
from wringlet import (
    data_provenance_enabled, 
    data_provenance_session_builder,
    # add_provenance_column, 
    # remove_provenance_column
)


In [4]:
from pyspark.sql import SparkSession


active_session = SparkSession.getActiveSession()
if active_session is not None:
    active_session.stop()

spark = (
    # SparkSession
    # .builder
    data_provenance_session_builder("semiwhy")
    .appName("data-provenance-notebook")
    .getOrCreate()
)


26/07/31 17:16:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


## Create toy dataframes

In [6]:
from datetime import date
df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.show(truncate=False)

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



+-------+----------+-----+--------+
|product|date      |price|quantity|
+-------+----------+-----+--------+
|A      |2026-01-15|10.0 |90      |
|A      |2026-01-16|10.0 |120     |
|A      |2026-01-17|5.0  |300     |
|B      |2026-01-15|100.0|20      |
|B      |2026-01-16|100.0|30      |
|C      |2026-01-17|80.0 |60      |
|F      |2026-01-16|50.0 |70      |
+-------+----------+-----+--------+



In [7]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["letter","labell"]
)

df2.printSchema()
df2.show(truncate=False)



root
 |-- letter: string (nullable = true)
 |-- labell: string (nullable = true)

+------+--------+
|letter|labell  |
+------+--------+
|A     |Bike    |
|B     |Handball|
|C     |Bike    |
|D     |Handball|
|E     |Running |
+------+--------+



In [8]:
from datetime import date

dft = spark.createDataFrame([
    ("A", date(2026, 1, 14), 10.0, 50),
    ("D", date(2026, 1, 15), 20.0, 70),
    ("C", date(2026, 1, 17), 80.0, 30),
    ("B", date(2026, 1, 18), 100.0, 10),
    ("F", date(2026, 1, 18), 80.0, 50),
    ("C", date(2026, 1, 15), 80.0, 60),
    ("F", date(2026, 1, 17), 75.0, 70)
], ["product", "date", "price", "quantity"]
)

dft.printSchema()
dft.show(truncate=False)

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)

+-------+----------+-----+--------+
|product|date      |price|quantity|
+-------+----------+-----+--------+
|A      |2026-01-14|10.0 |50      |
|D      |2026-01-15|20.0 |70      |
|C      |2026-01-17|80.0 |30      |
|B      |2026-01-18|100.0|10      |
|F      |2026-01-18|80.0 |50      |
|C      |2026-01-15|80.0 |60      |
|F      |2026-01-17|75.0 |70      |
+-------+----------+-----+--------+



## Tests with data_provenance_enabled
### Select

In [9]:
df.select("*").show(truncate=False)

+-------+----------+-----+--------+
|product|date      |price|quantity|
+-------+----------+-----+--------+
|A      |2026-01-15|10.0 |90      |
|A      |2026-01-16|10.0 |120     |
|A      |2026-01-17|5.0  |300     |
|B      |2026-01-15|100.0|20      |
|B      |2026-01-16|100.0|30      |
|C      |2026-01-17|80.0 |60      |
|F      |2026-01-16|50.0 |70      |
+-------+----------+-----+--------+



In [10]:
dft.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, df, df2, "sales") as (df_bis, df2_bis, dft_bis):
    print(df_bis.show(truncate=False))
    print(df2_bis.show(truncate=False))
    print(spark.table(dft_bis).show(truncate=False))


+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+------------------------------------+
|A      |2026-01-15|10.0 |90      |274dd17e-08a2-450d-b930-430fb0c3ecd5|
|A      |2026-01-16|10.0 |120     |e3333f6c-16a3-44c4-ac47-cceb23985eff|
|A      |2026-01-17|5.0  |300     |fea969b5-e009-43a8-a099-b6af93a8a7fa|
|B      |2026-01-15|100.0|20      |4412af7b-a042-4c97-a5d0-20baf77dc906|
|B      |2026-01-16|100.0|30      |1f1bf6ee-dd0b-49c4-b0e8-37bc5e864faa|
|C      |2026-01-17|80.0 |60      |ea507b34-2871-4ac2-8144-ee0ef034553e|
|F      |2026-01-16|50.0 |70      |37eeb82c-3eb7-44b0-a82e-95d94a9de707|
+-------+----------+-----+--------+------------------------------------+

None
+------+--------+------------------------------------+
|letter|labell  |_provenance_tag                     |
+------+--------+------------------------------------+
|A     |Bike    |6ed6e181-

In [11]:
dft.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, df, df2, "sales") as (df_bis, df2_bis, dft_bis):
    print(df_bis.select("*").filter(df_bis["product"] == "A").show(truncate=False), end="\n\n")
    print(df2_bis.select("*").filter(df2_bis["letter"] == "B").show(truncate=False), end="\n\n")
    print(spark.table(dft_bis).filter(spark.table(dft_bis)["product"] == "C").show(truncate=False), end="\n\n")



+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+------------------------------------+
|A      |2026-01-15|10.0 |90      |b3dc9157-d13c-42a7-a3c8-c855de2f0e42|
|A      |2026-01-16|10.0 |120     |5a539cda-8fe1-4afe-a9cc-ba8eba24b9b1|
|A      |2026-01-17|5.0  |300     |9d981b45-b381-4f71-a901-fde42e05b685|
+-------+----------+-----+--------+------------------------------------+

None

+------+--------+------------------------------------+
|letter|labell  |_provenance_tag                     |
+------+--------+------------------------------------+
|B     |Handball|d7648346-cc0c-4ed2-82b6-22f2739994af|
+------+--------+------------------------------------+

None

+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+-------------------

In [12]:
with data_provenance_enabled(spark, df) as (df_bis):
    df3 = df_bis.select("*")
df3.show(truncate=False)

+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+------------------------------------+
|A      |2026-01-15|10.0 |90      |87292335-7758-4a54-b8da-d2e254899a06|
|A      |2026-01-16|10.0 |120     |135007fd-a27b-4c73-aee0-edc1bb72212c|
|A      |2026-01-17|5.0  |300     |d4780b97-fdce-4637-8051-2da27b523975|
|B      |2026-01-15|100.0|20      |51e7010d-089f-4cbf-b0c2-d53471c2d2b1|
|B      |2026-01-16|100.0|30      |878c4c49-140e-4462-a5dc-0919c3cc20d4|
|C      |2026-01-17|80.0 |60      |e5b788db-6881-4662-b6c7-fe6b10e25f5c|
|F      |2026-01-16|50.0 |70      |cc06eed5-64fe-4e2d-8166-502772a9b972|
+-------+----------+-----+--------+------------------------------------+



In [13]:
with data_provenance_enabled(spark, df2) as (df2_bis):
    df4 = df2_bis.select("letter")
    df4.show(truncate=False)


+------+------------------------------------+
|letter|_provenance_tag                     |
+------+------------------------------------+
|A     |0ecaabcd-6e16-4bc2-8023-def666ec7c1d|
|B     |df9658af-efaf-441a-8a43-bdbe953c6930|
|C     |0cdf3485-232b-40bf-b01a-e860bad61b08|
|D     |a7f18064-961b-4341-a850-353a045077eb|
|E     |aafd4674-367c-4f2b-99bb-8aabe4075a20|
+------+------------------------------------+



In [14]:
# Some tags for df2 are the same as those for df
df.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, "sales"):
    res = spark.sql("select product from sales")
res.show(truncate=False)

+-------+------------------------------------+
|product|_provenance_tag                     |
+-------+------------------------------------+
|A      |a4c8437c-31ba-40e6-ba5f-78821f3645bf|
|A      |c1ca871e-61c8-4e83-9110-3312ebf29df0|
|A      |d78c99b1-849e-4b78-912a-d8a1bf2a6ac0|
|B      |a988a1a6-e658-48f9-97ac-b580f8126d98|
|B      |0a52ae90-8299-4a77-9f94-b07b0c67ada3|
|C      |4a2932ea-c090-4d9a-8370-c1d5155dfaf6|
|F      |4bc901d7-1fbd-4bc9-a95d-242e01d9f8e3|
+-------+------------------------------------+



In [15]:
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark,"category"):
    res2 = spark.sql("select letter from category")
res2.show(truncate=False)

+------+------------------------------------+
|letter|_provenance_tag                     |
+------+------------------------------------+
|A     |e7aa5afb-1a42-4885-af92-d41dbf802c56|
|B     |e63dbdef-63c4-48f6-91dd-bfc05f15edf5|
|C     |3d554ec0-6ff1-40ff-8cb0-526f9fc58315|
|D     |5621e01a-fc66-4b7e-8fe6-cc1efe455076|
|E     |7a8e154a-9f75-4867-ad41-12e3750dbad7|
+------+------------------------------------+



### Join

In [16]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark, "sales", "category") as (df_sales, df_category):
    res = spark.sql(
        f"select * from {df_sales} s join {df_category} c on s.product = c.letter"
    )
res.show(truncate=False)

+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|product|date      |price|quantity|letter|labell  |_provenance_tag                                                             |
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|A      |2026-01-15|10.0 |90      |A     |Bike    |[1df47d8a-5912-4088-aa69-7a985a65af94, 3cb15484-ffa8-4489-8a12-624505bfb2de]|
|A      |2026-01-16|10.0 |120     |A     |Bike    |[629a01b1-cb69-4fa6-80da-a1f9ea08a8e5, 3cb15484-ffa8-4489-8a12-624505bfb2de]|
|A      |2026-01-17|5.0  |300     |A     |Bike    |[214f420c-3725-4e53-a0cb-7d351f5bffc3, 3cb15484-ffa8-4489-8a12-624505bfb2de]|
|B      |2026-01-15|100.0|20      |B     |Handball|[03517d57-46a4-4ee1-be8d-4e11a850c0d1, 79aface1-ce97-49f5-8e32-aa2bb08113e4]|
|B      |2026-01-16|100.0|30      |B     |Handball|[9c8dfa89-9038-4bb3-8c56-274db741516d, 79aface

In [17]:
# Only the provenance tags of df is taken in account
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df5 = df_bis.select("*").join(df2_bis, df_bis.product == df2_bis.letter)

df5.show(truncate=False)

+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|product|date      |price|quantity|letter|labell  |_provenance_tag                                                             |
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|A      |2026-01-15|10.0 |90      |A     |Bike    |[e569b7a4-4f78-49ad-8734-72b43fb1a713, a6d03607-0cb0-4f2d-a162-a872db5e35cf]|
|A      |2026-01-16|10.0 |120     |A     |Bike    |[4c5e3264-ca53-4949-b89e-ce3e32504cd6, a6d03607-0cb0-4f2d-a162-a872db5e35cf]|
|A      |2026-01-17|5.0  |300     |A     |Bike    |[10f9ac03-6173-4b52-9373-6b891d0db0d5, a6d03607-0cb0-4f2d-a162-a872db5e35cf]|
|B      |2026-01-15|100.0|20      |B     |Handball|[4f947f1e-d46a-4595-943c-dfd716f5dd4f, 881b6ce4-c7ca-45d4-b7c5-ccd3e08781a9]|
|B      |2026-01-16|100.0|30      |B     |Handball|[f3068988-5b3c-43a2-8ae2-001477a0f56f, 881b6ce

In [18]:
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df6 = df_bis.join(df2_bis, df_bis.product == df2_bis.letter, "outer").select("product")

df6.show(truncate=False)

+-------+----------------------------------------------------------------------------+
|product|_provenance_tag                                                             |
+-------+----------------------------------------------------------------------------+
|A      |[ec33ea16-a3f7-4ada-b7f1-4dd47a194a3b, aca917ce-5800-4220-9936-105e901cc216]|
|A      |[106e8982-af4a-45c8-962c-ddefe7927420, aca917ce-5800-4220-9936-105e901cc216]|
|A      |[39f66554-6321-400c-8214-7adb4dcf200b, aca917ce-5800-4220-9936-105e901cc216]|
|B      |[501eabce-71f7-4d11-aabe-ca6f9ff606c2, dbb65bcb-e8eb-4170-a88e-97e704b2f222]|
|B      |[3c7236ac-ecf1-422a-a958-fd54c59f3710, dbb65bcb-e8eb-4170-a88e-97e704b2f222]|
|C      |[936f2e17-f53f-4480-88b3-2e8e4b5e7899, c9a7b9de-bc4f-491e-9dc0-c5b0d3a0fad2]|
|NULL   |[8a5429a3-c176-480d-93c6-2377103430d9]                                      |
|NULL   |[940df3a2-458e-4c9a-be9a-b823def71c03]                                      |
|F      |[dac54ced-7f3a-4884-9001-e8e5c08ca

In [19]:
# Same here, only the provenance tags of df are taken in account
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df7 = df_bis.select("*").join(df2_bis, df_bis.product==df2_bis.letter, "outer")
df7.show(truncate=False)

+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|product|date      |price|quantity|letter|labell  |_provenance_tag                                                             |
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|A      |2026-01-15|10.0 |90      |A     |Bike    |[fb05366b-9f21-49bd-b821-566b3cdd78af, e12e2979-09d5-4869-a9c1-9ed2408d8687]|
|A      |2026-01-16|10.0 |120     |A     |Bike    |[862b700b-3f34-4e56-8a81-1bc8a0aba777, e12e2979-09d5-4869-a9c1-9ed2408d8687]|
|A      |2026-01-17|5.0  |300     |A     |Bike    |[7256736a-52f4-401e-a7da-6c5ab8ab36e8, e12e2979-09d5-4869-a9c1-9ed2408d8687]|
|B      |2026-01-15|100.0|20      |B     |Handball|[a4e75d35-d91b-4a2e-a723-b252c14d3a44, 26a78af5-f728-4203-98ab-ec96e84c76e2]|
|B      |2026-01-16|100.0|30      |B     |Handball|[b389fba0-6ed1-4386-a21a-6111dc56e663, 26a78af

### Where

In [20]:
with data_provenance_enabled(spark, df) as (df_bis):
    df8 = df_bis.select("*").filter("price > 10")
df8.show(truncate=False)

+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+------------------------------------+
|B      |2026-01-15|100.0|20      |1b7c818c-8af2-4acc-84b0-2f201e312ef5|
|B      |2026-01-16|100.0|30      |ad946c5c-02af-4cdd-926a-eb4a77974b46|
|C      |2026-01-17|80.0 |60      |7ad94a22-2bb1-4dd9-a76c-308f986b9e6b|
|F      |2026-01-16|50.0 |70      |12213053-67b8-4297-8a18-02b4e3e6766e|
+-------+----------+-----+--------+------------------------------------+



In [21]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") as (df_sales):
    res = spark.sql("select product, price from sales where price>10").show(truncate=False)

res

+-------+-----+------------------------------------+
|product|price|_provenance_tag                     |
+-------+-----+------------------------------------+
|B      |100.0|aff33a26-fdb7-4809-b4ab-d4223e72f541|
|B      |100.0|32ff9acf-4ee5-4b52-9844-bf152da97712|
|C      |80.0 |430302c6-ae7f-47d7-b018-e2a346fdded3|
|F      |50.0 |52376ed0-8d5c-4519-a8ba-b947a21501c8|
+-------+-----+------------------------------------+



In [22]:
# With select statement
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df9 = df_bis.select("*").filter("price > 10").sort("price").join(df2_bis, df_bis.product == df2_bis.letter)
df9.show(truncate=False)

+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|product|date      |price|quantity|letter|labell  |_provenance_tag                                                             |
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|B      |2026-01-15|100.0|20      |B     |Handball|[5e3578cf-e51e-4ca5-83f5-5a482f47e604, fd14fc11-2d76-4fd2-bb64-69163147d62a]|
|B      |2026-01-16|100.0|30      |B     |Handball|[bbf1044c-832c-4289-a230-f5ccbebc1fa5, fd14fc11-2d76-4fd2-bb64-69163147d62a]|
|C      |2026-01-17|80.0 |60      |C     |Bike    |[88483a9e-7330-4e56-9c1f-6b37f8d7eedf, 3b92fb8d-7705-41d7-8a9f-a6590747d2e8]|
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+



In [23]:
# Without select statement
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df10 = df_bis.filter("price > 10").sort("price").join(df2_bis, df_bis.product == df2_bis.letter)
df10.show(truncate=False)

+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|product|date      |price|quantity|letter|labell  |_provenance_tag                                                             |
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+
|B      |2026-01-15|100.0|20      |B     |Handball|[dc2d582d-2615-4689-8057-80501f91cf0f, 8cc8ab76-b34b-49e2-a421-7fce933a2d1d]|
|B      |2026-01-16|100.0|30      |B     |Handball|[6f2e4e1d-3c20-42ad-9613-db2c18d998a3, 8cc8ab76-b34b-49e2-a421-7fce933a2d1d]|
|C      |2026-01-17|80.0 |60      |C     |Bike    |[ee9f0448-ea14-4d3c-8ce4-c08f37189aa0, 8fc008b7-4c33-4998-8959-a46583bf6f1f]|
+-------+----------+-----+--------+------+--------+----------------------------------------------------------------------------+



### Aggregation

In [24]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales"):
    res = spark.sql("select sum(quantity) from sales group by product").show(truncate=False)
res

+-------------+------------------------------------------------------------------------------------------------------------------+
|sum(quantity)|_provenance_tag                                                                                                   |
+-------------+------------------------------------------------------------------------------------------------------------------+
|510          |[ef4a54f5-767e-4785-af3c-e54c88ee57b2, 04de459c-57d3-44a5-b2f6-e1d3ea69a5ec, 1e47ab60-6472-4d05-8b69-2648925feedc]|
|50           |[c9da16b2-1106-4df1-9512-907df5318055, 37ec15bd-8e83-4e7e-ab23-fbf44bae9f15]                                      |
|60           |[9ea771f9-fa58-4369-a260-a45d0b9215db]                                                                            |
|70           |[4a7acef2-3290-4237-a7e9-db811c328360]                                                                            |
+-------------+--------------------------------------------------------------------

In [25]:
with data_provenance_enabled(spark, df) as (df_bis):
    df11 = df_bis.groupBy("product").agg({"quantity": "sum"})
df11.show(truncate=False)

+-------+-------------+------------------------------------------------------------------------------------------------------------------+
|product|sum(quantity)|_provenance_tag                                                                                                   |
+-------+-------------+------------------------------------------------------------------------------------------------------------------+
|A      |510          |[dec7657b-aa78-48a8-813e-4d9bc839ddf2, e1d8b5df-9a98-498a-bd25-0ac8ee86b046, 0c9956b4-6fe0-4b31-ad55-1a84c082fb66]|
|B      |50           |[36f9bc8a-956e-47e5-ab0b-5188f000ee3c, 6de4a811-6446-4268-8ef6-c76fd84ac25d]                                      |
|C      |60           |[c3fdffb2-d51f-4785-9b3b-22b6722b4cc4]                                                                            |
|F      |70           |[fba8c32b-86bc-47af-a947-75ee86f4b239]                                                                            |
+-------+-------------+----

In [26]:
with data_provenance_enabled(spark, df) as (df_bis):
    df12 = df_bis.withColumn("revenue", df_bis["quantity"] * df_bis["price"]) \
            .groupBy("product") \
            .agg({"revenue": "sum"})
df12.show(truncate=False)

+-------+------------+------------------------------------------------------------------------------------------------------------------+
|product|sum(revenue)|_provenance_tag                                                                                                   |
+-------+------------+------------------------------------------------------------------------------------------------------------------+
|A      |3600.0      |[021f5578-b1f5-418a-bf94-9edb685dc964, 8ad08e0d-0534-4e73-b711-12082ddbdd04, e564030a-f66c-4640-854b-686db32678b4]|
|B      |5000.0      |[525392c5-ab43-45ca-93e3-da0200e17f6a, 09c8375e-51f8-4a7f-afca-5a718bd71def]                                      |
|C      |4800.0      |[a03512cb-8986-4143-8bfa-c8e4ccc78675]                                                                            |
|F      |3500.0      |[20d39a38-d206-4a7d-8873-7480dbaeb67d]                                                                            |
+-------+------------+------------

In [27]:
with data_provenance_enabled(spark, df) as (df_bis):
    df13 = df_bis.groupBy("product").agg({"quantity": "sum"})
df13.show(truncate=False)

+-------+-------------+------------------------------------------------------------------------------------------------------------------+
|product|sum(quantity)|_provenance_tag                                                                                                   |
+-------+-------------+------------------------------------------------------------------------------------------------------------------+
|A      |510          |[7d586bb3-9cba-449f-8dfc-d3ca81804004, 5ef2e036-77a1-4e12-b26c-247bd4a72cb8, 7645e9ad-1550-4f04-844d-46e008142284]|
|B      |50           |[53278ea2-c384-49cf-ae2f-152322413333, 51771693-9319-4391-8625-70a6e7638312]                                      |
|C      |60           |[198ac971-3ba8-4df7-a8f7-3896167257ce]                                                                            |
|F      |70           |[2d8e6f80-c321-431f-9072-466e4cffa474]                                                                            |
+-------+-------------+----

In [28]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") :
    res = spark.sql("select distinct product from sales").show(truncate=False)

res

+-------+--------------------------------------+
|product|_provenance_tag                       |
+-------+--------------------------------------+
|A      |[828623bb-ac9c-4617-bc72-4623f1275cd0]|
|B      |[65a80d67-f73f-4370-8e21-9652bef2ba60]|
|C      |[6e4ef5f7-875d-4a99-bf54-ac5a7a8861d8]|
|F      |[5bdb7db8-917e-4778-96a6-e99a852b2a44]|
+-------+--------------------------------------+



In [29]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales"
) :
    res = spark.sql("select product from sales group by product").show(truncate=False)
res

+-------+------------------------------------------------------------------------------------------------------------------+
|product|_provenance_tag                                                                                                   |
+-------+------------------------------------------------------------------------------------------------------------------+
|A      |[9f2033a3-abbc-4c7a-973c-2aea0a4255ef, 9f9bf88c-64fb-4150-ac71-34762f991fff, f5ff792f-7132-42cf-9e51-2d26f3b25efa]|
|B      |[2a6933ff-5b08-44fa-b47a-8d65eec1e065, 7592fb83-f529-4c0a-bd99-27887cb4e9ef]                                      |
|C      |[0bfdf0e0-6cc5-43c8-81b1-f06357325cb1]                                                                            |
|F      |[06ce085d-8fcf-4145-b84e-f3cfea96ef92]                                                                            |
+-------+------------------------------------------------------------------------------------------------------------------+


In [30]:
with data_provenance_enabled(spark, df, dft) as (df_bis, dft_bis):
    df14 = df_bis.select("product").distinct().union(dft_bis.select("product").distinct())
df14.show(truncate=False)

+-------+--------------------------------------+
|product|_provenance_tag                       |
+-------+--------------------------------------+
|A      |[7eff6de2-9d7d-4dec-842f-90f8f3cd3507]|
|B      |[53f07daf-b428-4f2a-89d5-1953bddf3875]|
|C      |[bd926e31-59df-4d8f-97fc-640db0293966]|
|F      |[2d492d97-9674-4e1f-ac8e-fa206f1032f4]|
|A      |[7aa68d56-e90e-47c9-bf84-5dc4f4e5c721]|
|B      |[7231bf9d-8af2-4c8e-a22d-5593f967fe4c]|
|C      |[77acb976-324c-4036-938f-8d41cddd59d9]|
|D      |[8ef00556-8a5e-4fb6-b511-db0ad62d322a]|
|F      |[ebc2b9c7-ee15-495f-9d9b-4fbfb2684728]|
+-------+--------------------------------------+



In [31]:
# The provenance tags of df and dft are not always different
# The distinct operation does not work as expected, it does not always create new tags for the distinct values
# and it can reuse the same tags for the same values in df and dft

print(spark.conf.get("spark.provenance.enabled", "false"))
df15 = df_bis.select("product").distinct().union(dft_bis.select("product").distinct())
print(spark.conf.get("spark.provenance.enabled", "false"))
df15.show(truncate=False)

false
false
+-------+
|product|
+-------+
|A      |
|B      |
|C      |
|F      |
|A      |
|D      |
|C      |
|B      |
|F      |
+-------+



In [32]:
df.createOrReplaceTempView("sales1")
dft.createOrReplaceTempView("sales2")

with data_provenance_enabled(spark, "sales1", "sales2") as (df_sales1, df_sales2):
    res = spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2")
res.show(truncate=False)

+-------+--------------------------------------+
|product|_provenance_tag                       |
+-------+--------------------------------------+
|A      |[f2fc6cc1-a1b2-4f6d-8ebf-2c05bddda81f]|
|B      |[23c21db8-550e-4c7f-b604-4eea03085243]|
|C      |[68375eac-3696-41cd-a574-ee87e508fcb4]|
|D      |[e59e4a66-fd08-4653-bb67-5edb45489284]|
|F      |[c14bda7a-26dc-4487-a7df-3342be040fb1]|
+-------+--------------------------------------+



In [33]:
# Same exemple without provenance
df.createOrReplaceTempView("sales1")
dft.createOrReplaceTempView("sales2")
spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2").show(truncate=False)

+-------+
|product|
+-------+
|A      |
|B      |
|C      |
|F      |
|D      |
+-------+



In [34]:
df.createOrReplaceTempView("sales")
spark.sql("select * from sales").show(truncate=False)

+-------+----------+-----+--------+
|product|date      |price|quantity|
+-------+----------+-----+--------+
|A      |2026-01-15|10.0 |90      |
|A      |2026-01-16|10.0 |120     |
|A      |2026-01-17|5.0  |300     |
|B      |2026-01-15|100.0|20      |
|B      |2026-01-16|100.0|30      |
|C      |2026-01-17|80.0 |60      |
|F      |2026-01-16|50.0 |70      |
+-------+----------+-----+--------+



In [35]:
with data_provenance_enabled(spark, df) as (df_bis):
    # 1. Pipeline
    df8 = df_bis.select("*").filter("price > 10")
    
    # 2. On extrait les sources minimales DEPUIS df_bis tant que les tags sont actifs !
    minimal_sources = df8.get_minimal_sources([df_bis])

# 3. On affiche le résultat (en dehors du bloc avec succès)
initial_rows = minimal_sources[0]
initial_rows.show(truncate=False)

+-------+----------+-----+--------+
|product|date      |price|quantity|
+-------+----------+-----+--------+
|B      |2026-01-15|100.0|20      |
|F      |2026-01-16|50.0 |70      |
|B      |2026-01-16|100.0|30      |
|C      |2026-01-17|80.0 |60      |
+-------+----------+-----+--------+

